# Lesson 9.3: How Do You RAG Over APIs and Real-Time Data Sources?

**Companion notebook for Lesson 9.3**

---

| Section | What you will build |
|---|---|
| 1. Three Kinds of Knowledge | Prove the gap — same question, three very different answers from three sources |
| 2. Router with API Lane | Extend the 9.2 router to recognise freshness-demanding questions |
| 3. Tool Definitions | Write good tool specs; show how description quality determines selection accuracy |
| 4. API Tool Anatomy | Mock tool wrapper: validation → cache → retry → response shaping → error formatting |
| 5. ReAct Loop | Reason-Act-Observe: the LLM picks tools, your code runs them, results feed back |
| 6. Full Hybrid Pipeline | Vector + SQL + API working together on one question |
| 7. Rate Limiter | Sliding-window rate limiter with fallback hierarchy |
| 8. Three-Layer Cache | Result cache → semantic cache → negative cache |
| 9. Streaming Snapshot Pattern | Background worker + in-memory snapshot store — decouple freshness from latency |
| 10. Evaluation | 10 questions across all source types; source coverage chart |
| 11. Claude API | Real tool calling with Claude — tool_use content blocks, multi-turn loop |

**Required:** `sentence-transformers`, `numpy`, `matplotlib`  
**Built-in (no install):** `sqlite3`, `threading`, `time`, `json`  
**Optional (Section 11):** `anthropic`

> **Setup:** we re-use the same SQLite DB and four policy docs from Lesson 9.2,
> then add four mock API tools (shipping, weather, exchange-rate, service-status).
> No real API keys are needed until Section 11.


In [ ]:
# Uncomment to install
# !pip install sentence-transformers numpy matplotlib
# !pip install anthropic   # optional — Section 11

%matplotlib inline
import os, re, json, sqlite3, time, threading, warnings
import random
from datetime import datetime, timedelta
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

warnings.filterwarnings('ignore')
os.environ['OMP_NUM_THREADS']         = '1'
os.environ['MKL_NUM_THREADS']         = '1'
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '60'
os.environ['TOKENIZERS_PARALLELISM']  = 'false'

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size']      = 11
plt.rcParams['axes.grid']      = True
plt.rcParams['grid.alpha']     = 0.3
random.seed(42)
np.random.seed(42)

def show_plot():
    plt.tight_layout()
    plt.show()

print('Imports ready.')


In [ ]:
# ── Re-create the SQLite DB from Lesson 9.2 ─────────────────────────────────
conn = sqlite3.connect(':memory:')
conn.row_factory = sqlite3.Row

conn.executescript("""
CREATE TABLE warehouses (
    id INTEGER PRIMARY KEY, city TEXT, region TEXT,
    lat REAL, lon REAL
);
CREATE TABLE customers (
    id INTEGER PRIMARY KEY, name TEXT, email TEXT,
    region TEXT, signup_date TEXT, tier TEXT
);
CREATE TABLE orders (
    id INTEGER PRIMARY KEY, customer_id INTEGER,
    order_date TEXT, status TEXT, total_amount REAL,
    tracking_id TEXT, carrier TEXT, ship_date TEXT
);
CREATE TABLE exchange_rates_cache (
    pair TEXT PRIMARY KEY, rate REAL, updated_at TEXT
);
""")

WAREHOUSES = [
    (1, 'Los Angeles', 'West', 34.05, -118.24),
    (2, 'Chicago',     'Central', 41.88, -87.63),
    (3, 'New York',    'East', 40.71, -74.01),
    (4, 'Miami',       'South', 25.76, -80.19),
]
conn.executemany('INSERT INTO warehouses VALUES (?,?,?,?,?)', WAREHOUSES)

NAMES = ['Alice Kim','Bob Chen','Carol Diaz','Dan Patel','Eve Muller',
         'Frank Osei','Grace Liu','Hana Berg','Ivan Soto','Jess Park']
TIERS = ['free','pro','pro','enterprise']
REGIONS = ['North America','Europe','APAC']
CARRIERS = ['FedEx','UPS','USPS','DHL']
STATUSES = ['shipped','shipped','pending','refunded','cancelled']
TRACKING_PREFIXES = {'FedEx':'FX','UPS':'1Z','USPS':'94','DHL':'JD'}

base = datetime(2024, 1, 1)
for i, name in enumerate(NAMES, start=1):
    conn.execute(
        'INSERT INTO customers VALUES (?,?,?,?,?,?)',
        (i, name, name.lower().replace(' ','.')+"@co.com",
         random.choice(REGIONS),
         (base + timedelta(days=random.randint(0,500))).strftime('%Y-%m-%d'),
         random.choice(TIERS))
    )

for i in range(1, 21):
    cust_id  = random.randint(1, 10)
    carrier  = random.choice(CARRIERS)
    prefix   = TRACKING_PREFIXES[carrier]
    track_id = f'{prefix}{i:08d}'
    odate    = (datetime.now() - timedelta(days=random.randint(1,90))).strftime('%Y-%m-%d')
    sdate    = (datetime.now() - timedelta(days=random.randint(0,5))).strftime('%Y-%m-%d')
    conn.execute(
        'INSERT INTO orders VALUES (?,?,?,?,?,?,?,?)',
        (i, cust_id, odate, random.choice(STATUSES),
         round(random.uniform(49, 1200), 2),
         track_id, carrier, sdate)
    )

conn.executemany(
    'INSERT INTO exchange_rates_cache VALUES (?,?,?)',
    [('USD/EUR', 0.92, '2024-01-15 09:00'),
     ('USD/GBP', 0.79, '2024-01-15 09:00'),
     ('USD/JPY', 148.5,'2024-01-15 09:00')]
)
conn.commit()

def qry(sql, params=()):
    cur = conn.execute(sql, params)
    cols = [d[0] for d in cur.description]
    return [dict(zip(cols, r)) for r in cur.fetchall()]

print('DB ready:', qry('SELECT COUNT(*) n FROM orders')[0]['n'], 'orders,',
      qry('SELECT COUNT(*) n FROM customers')[0]['n'], 'customers')

# ── Policy documents ──────────────────────────────────────────────────────────
TEXT_DOCS = [
    {'id':'policy_returns','title':'Returns & Refunds Policy',
     'text':('Customers may return any product within 30 days for a full refund. '
             'Deliveries delayed more than 48 hours qualify for a 20% partial refund. '
             'Refunds are processed within 5-7 business days. '
             'Contact returns@co.com or use the self-service portal.')},
    {'id':'policy_shipping','title':'Shipping & Fulfillment',
     'text':('Standard US shipping costs $9.99 (5-7 days). Expedited 2-day costs $24.99. '
             'Orders over $150 ship free. International: buyer pays duties. '
             'All hardware ships from our nearest warehouse. '
             'Carrier selection is automatic based on delivery zone.')},
    {'id':'policy_sla','title':'Service Level Agreement',
     'text':('Pro tier: 99.5% uptime SLA, 24h support response. '
             'Enterprise tier: 99.9% uptime SLA, 4h response, named account manager. '
             'Scheduled maintenance windows: Sundays 02:00-04:00 UTC. '
             'Incident communications via status.co.com and email.')},
    {'id':'memo_ops','title':'Operations Memo — Weather Policy',
     'text':('Shipments from LA warehouse are held when sustained winds exceed 45 mph. '
             'Chicago warehouse follows standard winter protocol Nov-Mar: 2-day buffer added. '
             'Miami warehouse suspends during hurricane warnings. '
             'Affected customers receive automatic notifications.')},
]
print(f'Docs: {len(TEXT_DOCS)} policy documents')
print('Setup complete.')


In [ ]:
from sentence_transformers import SentenceTransformer, util

print('Loading all-MiniLM-L6-v2...')
embedder = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')

doc_embeds = embedder.encode(
    [d['text'] for d in TEXT_DOCS],
    convert_to_tensor=True, show_progress_bar=False)

def vector_retrieve(query, top_k=2):
    q = embedder.encode(query, convert_to_tensor=True, show_progress_bar=False)
    scores = util.cos_sim(q, doc_embeds)[0].cpu().numpy()
    idx    = np.argsort(scores)[::-1][:top_k]
    return [(TEXT_DOCS[i], float(scores[i])) for i in idx]

print('Embedder ready.')


---
## 1. Three Kinds of Knowledge — The Same Question, Three Different Answers

```
Question: "Is order #7 late, and what's our refund policy for delays?"

  Vector search  → policy doc:  '48h delay = 20% refund'   (static, yours)
  SQL            → order table: shipped 3 days ago, status='shipped'  (dynamic, yours)
  API            → carrier now: stuck in Memphis since yesterday  (live, not yours)

  No single source answers this. All three pieces are needed.
```

| Source | Freshness | Ownership | Right tool |
|---|---|---|---|
| Documents | Slow-changing | Yours | Vector search |
| Database | Fast-changing | Yours | SQL |
| External APIs | Real-time | Not yours | Tool calling |

**The key insight:** "If I refreshed my data warehouse right now, would the answer change?"  
If yes → API. If no → vector or SQL. "Who owns the truth?" If someone else → API.


In [ ]:
# Show what each source alone can and can't answer for a shipping delay question

ORDER_ID = 7
order = qry('SELECT * FROM orders WHERE id=?', (ORDER_ID,))
if not order:
    order = qry('SELECT * FROM orders LIMIT 1')
order = order[0]

# ── Source 1: Vector search ───────────────────────────────────────────────────
q = f'What is the refund policy for a delayed order #{ORDER_ID}?'
doc, score = vector_retrieve(q)[0]

# ── Source 2: SQL ─────────────────────────────────────────────────────────────
days_since_ship = (datetime.now() - datetime.strptime(order['ship_date'], '%Y-%m-%d')).days

# ── Source 3: API (mock — live status would come from carrier) ────────────────
MOCK_SHIPPING_SNAPSHOTS = {
    order['tracking_id']: {
        'status':     'In Transit — Delayed',
        'location':   'Memphis, TN',
        'eta':        (datetime.now() + timedelta(days=1)).strftime('%Y-%m-%d'),
        'is_delayed': True,
        'delay_hours': 27,
    }
}
live_status = MOCK_SHIPPING_SNAPSHOTS.get(order['tracking_id'],
              {'status':'Unknown','is_delayed':False,'delay_hours':0})

print(f'Question: "{q}"\n')
print('─── Source 1: Vector Search ────────────────────────────────────')
print(f'  Doc [{doc["id"]}]: "{doc["text"][:120]}..."')
print(f'  Can answer: refund policy rule  |  Cannot answer: IS this order actually delayed?')

print()
print('─── Source 2: SQL ─────────────────────────────────────────────')
print(f'  Order #{ORDER_ID}: status={order["status"]}, carrier={order["carrier"]}, '
      f'tracking={order["tracking_id"]}, shipped {days_since_ship}d ago')
print(f'  Can answer: order facts  |  Cannot answer: WHERE is the package right now?')

print()
print('─── Source 3: Shipping API (live) ────────────────────────────')
print(f'  {live_status}')
print(f'  Can answer: live location & delay  |  Cannot answer: what refund does the customer get?')

print()
print('─── Synthesis (all three together) ───────────────────────────')
delay_h = live_status['delay_hours']
refund = '20%' if delay_h >= 48 else 'not yet eligible (< 48h delay)'
print(f'  Your order #{ORDER_ID} is stuck in {live_status["location"]} and is {delay_h}h late.')
print(f'  Per policy, a delay > 48h earns a 20% refund. Current refund eligibility: {refund}.')
print(f'  ETA updated to {live_status["eta"]}.')


---
## 2. The Router Grows Up — Adding the API Lane

The router from Lesson 9.2 knew three categories: `vector`, `sql`, `hybrid`.  
We add `api` and `api+hybrid` to handle freshness-demanding questions.

**Signal words for the API lane:**  
`right now`, `current`, `live`, `today's`, `is ... up`, `tracking`, `status of`,
`weather`, `exchange rate`, `stock price`, `where is my`

The key signal is **temporal urgency** — the user is asking about *this moment*, not history.


In [ ]:
API_SIGNALS = [
    r'right now', r'current(ly)?', r'live', r'today.s',
    r'is .+ up', r'is .+ down', r'service status', r'outage',
    r'track(ing)?', r'where is my', r'where.s my',
    r'weather', r'exchange rate', r'fx rate', r'stock price',
    r'package', r'shipment', r'delivery status',
]

SQL_SIGNALS = [
    r'how many', r'how much', r'count', r'total', r'average',
    r'top \d+', r'list all', r'last \d+ days?', r'last (week|month)',
    r'revenue', r'orders?', r'customers?', r'signed up',
]

VECTOR_SIGNALS = [
    r'policy', r'how does', r'explain', r'why', r'what is our',
    r'what are the', r'describe', r'difference between',
    r'sla', r'refund rule', r'cancellation',
]

def router(question):
    q   = question.lower()
    api = any(re.search(p, q) for p in API_SIGNALS)
    sql = any(re.search(p, q) for p in SQL_SIGNALS)
    vec = any(re.search(p, q) for p in VECTOR_SIGNALS)

    sources = []
    if api: sources.append('api')
    if sql: sources.append('sql')
    if vec: sources.append('vector')

    if len(sources) >= 2:
        return 'hybrid'
    return sources[0] if sources else 'vector'


ROUTER_TESTS = [
    ("What's our cancellation policy?",                              'vector'),
    ("How many users signed up yesterday?",                          'sql'),
    ("Is the payment gateway up right now?",                         'api'),
    ("What's the weather at our Boston warehouse?",                   'api'),
    ("Where is my package with tracking number FX00000007?",         'api'),
    ("What's the current USD/EUR exchange rate?",                    'api'),
    ("Which invoices are overdue and what's the USD/EUR rate?",      'hybrid'),
    ("Why was order #7 late and what's our delay refund policy?",    'hybrid'),
    ("How many shipped orders do we have?",                          'sql'),
    ("What are enterprise SLA guarantees?",                          'vector'),
]

print(f'{"Question":<55} {"Expected":<10} {"Router"}')
print('-' * 82)
correct = 0
for q, expected in ROUTER_TESTS:
    got  = router(q)
    mark = 'OK' if got == expected else '!'
    if got == expected:
        correct += 1
    print(f'{q:<55} {expected:<10} {got:<10} {mark}')
print(f'\nAccuracy: {correct}/{len(ROUTER_TESTS)}')


---
## 3. Tool Definitions — Descriptions Are Everything

The LLM picks tools based on their descriptions. Vague description = wrong tool called.

> **Test:** Read each description out loud. Could a new engineer figure out
> exactly when to use it without asking you? If not, your LLM can't either.

Two rules for good tool descriptions:
1. Say **when to use** (and optionally when NOT to)
2. Give **a concrete example** of the trigger phrase


In [ ]:
TOOL_SPECS = [
    {
        'name': 'get_shipping_status',
        'description': (
            'Get the CURRENT, LIVE shipping status and location for a specific order '
            'by its tracking ID. Use when the user asks where their package is RIGHT NOW, '
            'about delivery status, whether a shipment is delayed, or its ETA. '
            'Do NOT use for historical order data (use SQL) or refund policies (use vector).'
        ),
        'parameters': {
            'tracking_id': {'type': 'string', 'required': True,
                            'description': 'The carrier tracking number (e.g. FX00000007, 1Z00000012)'},
        },
        'returns': {'status': str, 'location': str, 'eta': str, 'is_delayed': bool, 'delay_hours': int},
        'cache_ttl_seconds': 60,
    },
    {
        'name': 'get_weather',
        'description': (
            'Get current weather conditions for a city. Use when the user asks about '
            'today\'s weather, whether weather might affect shipments or warehouse operations, '
            'or if conditions are safe for shipping. '
            'Do NOT use for historical weather data.'
        ),
        'parameters': {
            'city': {'type': 'string', 'required': True,
                     'description': 'City name (e.g. "Los Angeles", "Chicago")'},
        },
        'returns': {'condition': str, 'temp_f': float, 'wind_mph': float,
                    'is_shipping_safe': bool, 'advisory': str},
        'cache_ttl_seconds': 3600,
    },
    {
        'name': 'get_exchange_rate',
        'description': (
            'Get the CURRENT exchange rate between two currencies. '
            'Use when the user needs to convert amounts between currencies or asks for '
            'today\'s rate. Supported pairs: USD/EUR, USD/GBP, USD/JPY. '
            'Do NOT use for historical rates or rate trends.'
        ),
        'parameters': {
            'pair': {'type': 'string', 'required': True,
                     'description': 'Currency pair as "FROM/TO" (e.g. "USD/EUR")'},
        },
        'returns': {'pair': str, 'rate': float, 'updated_at': str},
        'cache_ttl_seconds': 30,
    },
    {
        'name': 'get_service_status',
        'description': (
            'Check if a TechCo service or API endpoint is currently UP or DOWN. '
            'Use when the user asks if a service is working, reports an outage, or '
            'asks about current system health. Services: "api", "dashboard", "checkout". '
            'Do NOT use for planned maintenance windows (use vector/SLA docs).'
        ),
        'parameters': {
            'service': {'type': 'string', 'required': True,
                        'description': 'Service name: "api", "dashboard", or "checkout"'},
        },
        'returns': {'service': str, 'status': str, 'latency_ms': int,
                    'last_incident': str},
        'cache_ttl_seconds': 15,
    },
]

print('=== Tool Registry ===\n')
for t in TOOL_SPECS:
    params = ', '.join(t['parameters'].keys())
    print(f'  {t["name"]}({params})')
    print(f'    TTL: {t["cache_ttl_seconds"]}s   Returns: {list(t["returns"].keys())}')
    print()

# ── Description quality demo: which tool gets selected for each question? ─────
TOOL_DESCS    = [t['description'] for t in TOOL_SPECS]
TOOL_NAMES    = [t['name'] for t in TOOL_SPECS]
tool_desc_emb = embedder.encode(TOOL_DESCS, convert_to_tensor=True, show_progress_bar=False)

def select_tool(question, top_k=1):
    q_emb  = embedder.encode(question, convert_to_tensor=True, show_progress_bar=False)
    scores = util.cos_sim(q_emb, tool_desc_emb)[0].cpu().numpy()
    idx    = np.argsort(scores)[::-1][:top_k]
    return [(TOOL_NAMES[i], float(scores[i])) for i in idx]

tool_selection_demos = [
    ('Where is my package FX00000007?',           'get_shipping_status'),
    ('Is it safe to ship from LA today?',          'get_weather'),
    ("What's the current USD/EUR rate?",           'get_exchange_rate'),
    ('Is the checkout service down right now?',    'get_service_status'),
]

print('=== Tool Selection by Description Similarity ===\n')
print(f'{"Question":<48} {"Expected":<25} {"Selected"}')
print('-' * 90)
for q, expected in tool_selection_demos:
    selected, score = select_tool(q)[0]
    mark = 'OK' if selected == expected else '!'
    print(f'{q:<48} {expected:<25} {selected:<25} score={score:.3f} {mark}')


---
## 4. API Tool Anatomy — The Five Layers

A production tool wrapper is not just an HTTP call. Between the LLM's request
and the API's response, you need:

```
LLM request
    │
    ▼  1. Input validation    — the LLM might hallucinate a malformed argument
    ▼  2. Cache check         — same args within TTL? Return cached result
    ▼  3. API call + retry    — with timeout and exponential backoff
    ▼  4. Response shaping    — drop irrelevant fields; return only what answers the question
    ▼  5. Error formatting    — structured errors with fallback_suggestion for the LLM
    │
LLM receives shaped result
```

**Why shape the response?** A typical shipping API returns 200+ fields.
The LLM will spend tokens on irrelevant data and may latch onto the wrong field.
Return only what answers the likely question.

**Why structure errors?** `ECONNREFUSED 10.0.0.4:443` is a dead end for the LLM.
`{"fallback_suggestion": "Check orders table for last known status"}` is an instruction.
Treat error responses as guidance, not just diagnostics.


In [ ]:
class ToolCache:
    """Simple TTL cache: stores (value, timestamp) per key."""

    def __init__(self, ttl_seconds):
        self.ttl  = ttl_seconds
        self._store = {}

    def get(self, key):
        entry = self._store.get(key)
        if entry and time.time() - entry['t'] < self.ttl:
            return entry['v']
        return None

    def set(self, key, value):
        self._store[key] = {'v': value, 't': time.time()}

    def stats(self):
        return {'entries': len(self._store), 'ttl': self.ttl}


# ── Mock data the tools will return ──────────────────────────────────────────
MOCK_SHIPPING_DB = {
    'FX00000007': {'status':'In Transit — Delayed','location':'Memphis, TN',
                   'eta':'2024-02-20','is_delayed':True,'delay_hours':27},
    '1Z00000012': {'status':'Out for Delivery','location':'Chicago, IL',
                   'eta':'2024-02-18','is_delayed':False,'delay_hours':0},
    'JD00000003': {'status':'Delivered','location':'New York, NY',
                   'eta':'2024-02-17','is_delayed':False,'delay_hours':0},
}

MOCK_WEATHER_DB = {
    'Los Angeles': {'condition':'Sunny','temp_f':78,'wind_mph':8,
                    'is_shipping_safe':True,'advisory':'Clear conditions.'},
    'Chicago':     {'condition':'Heavy Snow','temp_f':18,'wind_mph':32,
                    'is_shipping_safe':False,'advisory':'Winter protocol active — add 2-day buffer.'},
    'New York':    {'condition':'Cloudy','temp_f':45,'wind_mph':15,
                    'is_shipping_safe':True,'advisory':'Normal operations.'},
    'Miami':       {'condition':'Partly Cloudy','temp_f':83,'wind_mph':12,
                    'is_shipping_safe':True,'advisory':'Normal operations.'},
    'Boston':      {'condition':'Rain','temp_f':42,'wind_mph':20,
                    'is_shipping_safe':True,'advisory':'Minor delays possible.'},
}

MOCK_FX_DB = {
    'USD/EUR': 0.9187, 'USD/GBP': 0.7862, 'USD/JPY': 149.32,
}

MOCK_SERVICE_DB = {
    'api':       {'status':'operational','latency_ms':42,'last_incident':'None in last 30 days'},
    'dashboard': {'status':'degraded',   'latency_ms':890,'last_incident':'2024-02-18 14:22 UTC'},
    'checkout':  {'status':'operational','latency_ms':67,'last_incident':'None in last 30 days'},
}


class BaseTool:
    name        = ''
    description = ''
    cache_ttl   = 60

    def __init__(self):
        self.cache       = ToolCache(self.cache_ttl)
        self.call_count  = 0
        self.cache_hits  = 0

    def _validate(self, **kwargs):
        """Override to validate inputs. Return error dict or None."""
        return None

    def _fetch(self, **kwargs):
        """Override with actual API call / mock. Return shaped dict."""
        raise NotImplementedError

    def __call__(self, **kwargs):
        # Layer 1: Validate
        err = self._validate(**kwargs)
        if err:
            return err

        # Layer 2: Cache
        cache_key = json.dumps(kwargs, sort_keys=True)
        cached    = self.cache.get(cache_key)
        if cached:
            self.cache_hits += 1
            return {**cached, '_source': 'cache'}

        # Layer 3: Fetch with simulated retry
        self.call_count += 1
        result = self._fetch(**kwargs)

        # Layer 4+5: Shaping and error formatting happen inside _fetch
        if 'error' not in result:
            self.cache.set(cache_key, result)

        return {**result, '_source': 'live'}


class ShippingTool(BaseTool):
    name      = 'get_shipping_status'
    cache_ttl = 60

    def _validate(self, tracking_id=''):
        if not tracking_id or not re.match(r'^[A-Z0-9]{2,20}$', tracking_id.upper()):
            return {'error': 'invalid_tracking_id',
                    'user_message': 'That does not look like a valid tracking number.',
                    'fallback_suggestion': 'Ask the customer for the tracking number from their shipment email.'}
        return None

    def _fetch(self, tracking_id=''):
        data = MOCK_SHIPPING_DB.get(tracking_id.upper())
        if data:
            return dict(data)
        return {'error': 'tracking_not_found',
                'user_message': f'No tracking info found for {tracking_id}.',
                'fallback_suggestion': 'Check the orders table for the carrier and ship date.'}


class WeatherTool(BaseTool):
    name      = 'get_weather'
    cache_ttl = 3600

    def _validate(self, city=''):
        if not city or len(city) < 2:
            return {'error': 'invalid_city',
                    'user_message': 'Please provide a valid city name.',
                    'fallback_suggestion': 'Ask the user which city they mean.'}
        return None

    def _fetch(self, city=''):
        data = MOCK_WEATHER_DB.get(city)
        if data:
            return dict(data)
        return {'error': 'city_not_found',
                'user_message': f'Weather data not available for {city}.',
                'fallback_suggestion': 'Try a nearby major city or check a public weather service.'}


class ExchangeRateTool(BaseTool):
    name      = 'get_exchange_rate'
    cache_ttl = 30

    def _validate(self, pair=''):
        if not re.match(r'^[A-Z]{3}/[A-Z]{3}$', pair.upper()):
            return {'error': 'invalid_pair',
                    'user_message': 'Currency pair must be formatted as FROM/TO (e.g. USD/EUR).',
                    'fallback_suggestion': 'Confirm the currency codes with the user.'}
        return None

    def _fetch(self, pair=''):
        rate = MOCK_FX_DB.get(pair.upper())
        if rate:
            return {'pair': pair.upper(), 'rate': rate,
                    'updated_at': datetime.utcnow().strftime('%Y-%m-%d %H:%M UTC')}
        return {'error': 'pair_not_supported',
                'user_message': f'{pair} is not a supported currency pair.',
                'fallback_suggestion': f'Supported pairs: {list(MOCK_FX_DB.keys())}.'}


class ServiceStatusTool(BaseTool):
    name      = 'get_service_status'
    cache_ttl = 15

    def _validate(self, service=''):
        if service not in ('api', 'dashboard', 'checkout'):
            return {'error': 'unknown_service',
                    'user_message': f'{service} is not a monitored service.',
                    'fallback_suggestion': 'Available services: api, dashboard, checkout.'}
        return None

    def _fetch(self, service=''):
        data = MOCK_SERVICE_DB.get(service, {})
        return {'service': service, **data}


TOOL_REGISTRY = {
    'get_shipping_status': ShippingTool(),
    'get_weather':         WeatherTool(),
    'get_exchange_rate':   ExchangeRateTool(),
    'get_service_status':  ServiceStatusTool(),
}

print('=== Tool demo: each layer in action ===\n')

demos = [
    ('get_shipping_status', {'tracking_id': 'FX00000007'}, 'Valid — should succeed'),
    ('get_shipping_status', {'tracking_id': 'INVALID!@#'}, 'Malformed — validation layer catches it'),
    ('get_weather',         {'city': 'Chicago'},           'Valid — returns unsafe advisory'),
    ('get_exchange_rate',   {'pair': 'USD/EUR'},           'Valid — returns rate'),
    ('get_exchange_rate',   {'pair': 'USD/EUR'},           'Repeated — should return from cache'),
    ('get_service_status',  {'service': 'dashboard'},      'Valid — returns degraded status'),
]

for tool_name, args, note in demos:
    tool   = TOOL_REGISTRY[tool_name]
    result = tool(**args)
    source = result.pop('_source', 'live')
    print(f'[{tool_name}] {args}  ({note})')
    print(f'  source={source}  result={result}')
    print()

print('Cache stats:')
for name, tool in TOOL_REGISTRY.items():
    print(f'  {name}: {tool.call_count} live calls, {tool.cache_hits} cache hits')


---
## 5. The ReAct Loop — Reason, Act, Observe

```
                   ┌──────────────────────────────────┐
  User question ──►│  LLM: which tool? with what args? │
                   └─────────────┬────────────────────┘
                                 │  Tool call spec
                                 ▼
                   ┌──────────────────────────────────┐
                   │  Your code: execute the tool      │
                   └─────────────┬────────────────────┘
                                 │  Shaped result
                                 ▼
                   ┌──────────────────────────────────┐
                   │  LLM: more tools needed? or done? │◄── loop
                   └─────────────┬────────────────────┘
                                 │  Final answer
                                 ▼
                            User sees answer
```

The LLM is the **brain** (decides what to call, with what args).  
Your code is the **hands** (executes, handles auth, rate limits, retries).  
This separation keeps secrets out of the LLM's view and makes the system reliable.


In [ ]:
class MockReActLLM:
    """
    Simulates the LLM's role in a ReAct loop.
    Given a question and available tools, decides what to call.
    Returns a list of (tool_name, args) to execute, then a final answer.
    Replace with a real LLM in Section 11.
    """

    def decide(self, question, tool_results=None):
        """
        Returns: {'action': 'call_tools', 'calls': [...]} or
                 {'action': 'answer',     'text': '...'}
        """
        q  = question.lower()
        tr = tool_results or {}

        # ── Already have all data: synthesize answer ─────────────────────────
        if tr:
            parts = []
            for tool_name, result in tr.items():
                if 'error' in result:
                    parts.append(f'{tool_name} failed: {result.get("user_message","unavailable")}')
                elif tool_name == 'get_shipping_status':
                    delayed = result.get('is_delayed', False)
                    parts.append(
                        f"Your shipment is currently {result['status']} in {result['location']}. "
                        f"ETA: {result['eta']}."
                        + (f" It is {result['delay_hours']}h late." if delayed else '')
                    )
                elif tool_name == 'get_weather':
                    safe = result.get('is_shipping_safe', True)
                    parts.append(
                        f"Weather in {result.get('condition','unknown')}, "
                        f"{result.get('temp_f','?')}°F, "
                        f"winds {result.get('wind_mph','?')} mph. "
                        f"Shipping {'safe' if safe else 'NOT recommended'}. "
                        f"{result.get('advisory','')}"
                    )
                elif tool_name == 'get_exchange_rate':
                    parts.append(
                        f"Current {result['pair']} rate: {result['rate']} (as of {result['updated_at']})."
                    )
                elif tool_name == 'get_service_status':
                    status = result.get('status', 'unknown')
                    parts.append(
                        f"{result['service'].title()} service is {status}. "
                        f"Latency: {result.get('latency_ms','?')}ms. "
                        f"Last incident: {result.get('last_incident','none')}."
                    )
            return {'action': 'answer', 'text': ' '.join(parts)}

        # ── No results yet: decide which tools to call ────────────────────────
        calls = []
        if re.search(r'fx\d+|tracking|package|where is|shipment', q):
            track_match = re.search(r'(FX|1Z|94|JD)\d+', question, re.IGNORECASE)
            if track_match:
                calls.append(('get_shipping_status', {'tracking_id': track_match.group(0).upper()}))

        if re.search(r'weather|safe to ship|wind|temperature', q):
            city_match = re.search(
                r'(Los Angeles|Chicago|New York|Miami|Boston|LA)', question, re.IGNORECASE)
            city = city_match.group(0) if city_match else 'Los Angeles'
            if city.upper() == 'LA':
                city = 'Los Angeles'
            calls.append(('get_weather', {'city': city}))

        if re.search(r'exchange rate|usd|eur|gbp|jpy|currency|convert', q):
            pair_match = re.search(r'(USD|EUR|GBP|JPY)/(USD|EUR|GBP|JPY)', question, re.IGNORECASE)
            pair = pair_match.group(0).upper() if pair_match else 'USD/EUR'
            calls.append(('get_exchange_rate', {'pair': pair}))

        if re.search(r'service.*up|service.*down|is .+ up|outage|status|checkout|dashboard', q):
            svc_match = re.search(r'(api|dashboard|checkout)', q)
            svc = svc_match.group(0) if svc_match else 'api'
            calls.append(('get_service_status', {'service': svc}))

        if calls:
            return {'action': 'call_tools', 'calls': calls}

        return {'action': 'answer', 'text': 'I need more information to answer that question.'}


react_llm = MockReActLLM()


def react_loop(question, max_steps=3, verbose=True):
    """Run the full ReAct loop for a question."""
    tool_results = {}

    for step in range(1, max_steps + 1):
        decision = react_llm.decide(question, tool_results if step > 1 else None)

        if decision['action'] == 'answer':
            if verbose:
                print(f'  [Step {step}] ANSWER: {decision["text"]}')
            return decision['text']

        if verbose:
            print(f'  [Step {step}] CALL TOOLS: {[c[0] for c in decision["calls"]]}')

        for tool_name, args in decision['calls']:
            tool          = TOOL_REGISTRY[tool_name]
            result        = tool(**args)
            result.pop('_source', None)
            tool_results[tool_name] = result
            if verbose:
                print(f'  [Tool] {tool_name}({args}) → {result}')

    return 'Could not resolve after max steps.'


print('=== ReAct Loop Demos ===\n')
react_demos = [
    'Where is my package with tracking number FX00000007?',
    'Is it safe to ship from LA today?',
    'Is the checkout service up right now?',
]
for q in react_demos:
    print(f'Q: {q}')
    react_loop(q, verbose=True)
    print()


---
## 6. Full Hybrid Pipeline — Vector + SQL + API

All three sources working together on one question:

> *"Why is order FX00000007 late, and what's our refund policy for delays?"*

- **Vector** → policy doc: 48h delay = 20% refund  
- **SQL** → order record: tracking ID, carrier, ship date  
- **API** → live status: stuck in Memphis, 27h late  


In [ ]:
def hybrid_rag(question, verbose=True):
    """
    Full three-source hybrid RAG:
      1. Route
      2a. Vector path
      2b. SQL path
      2c. API path (ReAct loop)
      3. Synthesise
    """
    route = router(question)
    if verbose:
        print(f'[Router] {route.upper()}')

    doc_context = None
    sql_result  = None
    api_result  = {}

    # ── Vector ───────────────────────────────────────────────────────────────
    if route in ('vector', 'hybrid'):
        docs = vector_retrieve(question, top_k=2)
        doc_context = ' '.join(d['text'][:200] for d, _ in docs)
        if verbose:
            print(f'[Vector] Retrieved [{docs[0][0]["id"]}] score={docs[0][1]:.3f}')

    # ── SQL ──────────────────────────────────────────────────────────────────
    if route in ('sql', 'hybrid'):
        track_match = re.search(r'(FX|1Z|94|JD)\d+', question, re.IGNORECASE)
        if track_match:
            rows = qry('SELECT * FROM orders WHERE tracking_id=?',
                       (track_match.group(0).upper(),))
            sql_result = rows[0] if rows else None
            if verbose:
                print(f'[SQL] Order found: {sql_result}')

    # ── API ───────────────────────────────────────────────────────────────────
    if route in ('api', 'hybrid'):
        decision = react_llm.decide(question)
        if decision['action'] == 'call_tools':
            for tool_name, args in decision['calls']:
                result = TOOL_REGISTRY[tool_name](**args)
                result.pop('_source', None)
                api_result[tool_name] = result
                if verbose:
                    print(f'[API] {tool_name}({args}) → {result}')

    # ── Synthesise ────────────────────────────────────────────────────────────
    answer_parts = []

    if api_result.get('get_shipping_status'):
        s = api_result['get_shipping_status']
        if 'error' not in s:
            answer_parts.append(
                f"Live status: {s['status']} in {s['location']}, ETA {s['eta']}."
                + (f" Currently {s['delay_hours']}h delayed." if s.get('is_delayed') else '')
            )

    if sql_result:
        answer_parts.append(
            f"Order details: carrier={sql_result['carrier']}, "
            f"shipped={sql_result['ship_date']}, status={sql_result['status']}."
        )

    if doc_context:
        refund_match = re.search(r'delayed? more than (\d+) hours?.*?(\d+)%', doc_context)
        if refund_match:
            threshold = int(refund_match.group(1))
            pct       = int(refund_match.group(2))
            delay_h   = api_result.get('get_shipping_status', {}).get('delay_hours', 0)
            eligible  = delay_h >= threshold
            answer_parts.append(
                f"Refund policy: delays >{threshold}h = {pct}% refund. "
                f"This order ({delay_h}h late) is {'ELIGIBLE' if eligible else 'not yet eligible'}."
            )

    answer = ' | '.join(answer_parts) if answer_parts else 'Insufficient data.'
    if verbose:
        print(f'[Synthesis] {answer}')
    return {'route': route, 'answer': answer}


print('=== Full Hybrid Pipeline ===\n')
hybrid_demos = [
    'Why is order FX00000007 late, and what is our refund policy for delays?',
    'Is it safe to ship from Los Angeles today?',
    'Is the checkout service up right now?',
    "What's our cancellation policy?",
]
for q in hybrid_demos:
    print(f'\nQ: {q}')
    hybrid_rag(q, verbose=True)
    print()


---
## 7. Rate Limiter — You Will Hit Rate Limits. Plan for It.

Sliding-window rate limiter with a graceful fallback hierarchy:

```
1. Serve from cache    (even slightly stale > failing)
2. Fall back to SQL    (last known status from our DB)
3. Tell user clearly   ("Live tracking briefly unavailable")
```

**Critical rule:** Never let a rate-limit error become an LLM hallucination.  
If the API fails, the tool returns `{"error": "..."}`.  
Your synthesis prompt must say: *"If a tool returned an error, say so. Do not fabricate a value."*  
This is the most important rule of API-RAG.


In [ ]:
class RateLimiter:
    """Sliding-window rate limiter."""

    def __init__(self, max_calls, per_seconds):
        self.max_calls   = max_calls
        self.per_seconds = per_seconds
        self.calls       = []
        self.rejected    = 0

    def allow(self):
        now         = time.time()
        self.calls  = [t for t in self.calls if now - t < self.per_seconds]
        if len(self.calls) >= self.max_calls:
            self.rejected += 1
            return False
        self.calls.append(now)
        return True

    def status(self):
        now = time.time()
        active = [t for t in self.calls if now - t < self.per_seconds]
        return {'active': len(active), 'max': self.max_calls,
                'window_s': self.per_seconds, 'rejected': self.rejected}


shipping_limiter = RateLimiter(max_calls=5, per_seconds=10)

def rate_limited_shipping(tracking_id):
    """Tool call with rate limiting and fallback hierarchy."""
    if shipping_limiter.allow():
        result = TOOL_REGISTRY['get_shipping_status'](tracking_id=tracking_id)
        result.pop('_source', None)
        return result

    # Fallback 1: cache
    cache_key = json.dumps({'tracking_id': tracking_id})
    cached    = TOOL_REGISTRY['get_shipping_status'].cache.get(cache_key)
    if cached:
        return {**cached, '_note': 'served from cache (rate limited)'}

    # Fallback 2: SQL last-known status
    rows = qry('SELECT status, ship_date, carrier FROM orders WHERE tracking_id=?',
               (tracking_id,))
    if rows:
        return {'status': f"Last known (DB): {rows[0]['status']}",
                'location': 'unknown — live tracking unavailable',
                'eta': 'unknown', 'is_delayed': None,
                '_note': 'rate limited — serving SQL fallback'}

    # Fallback 3: graceful degradation
    return {'error': 'rate_limit_exceeded',
            'user_message': 'Live tracking is briefly unavailable. Please try again in a moment.',
            'fallback_suggestion': 'Check the orders table for the last known status.'}


print('=== Rate limiter demo (5 calls / 10s window) ===\n')
TRACK_ID = 'FX00000007'
for i in range(1, 9):
    result = rate_limited_shipping(TRACK_ID)
    status_str = shipping_limiter.status()
    note   = result.get('_note', result.get('error', 'OK'))
    print(f'  Call {i}: {note}  |  limiter={status_str}')


---
## 8. Three-Layer Cache

| Layer | What it caches | Key | TTL |
|---|---|---|---|
| Result cache | Raw API call result | `(tool_name, args)` | Tool-specific (15s–3600s) |
| Semantic cache | Full question answer | Embedding similarity | 30–60s |
| Negative cache | Error responses | `(tool_name, args)` | 5–30s |

**Rule of thumb:** If your tool is called more than once per minute on the same args,
you need Layer 1. If more than once per second, you need all three.

**The semantic cache trick:** "What's the weather in NYC?", "NYC weather right now?",
and "How's it looking in New York?" are the same question. Embedding similarity catches
this; exact-string matching never would. A 60-second semantic cache often cuts API
costs by 90%+ with no user-visible degradation.


In [ ]:
# ── Layer 2: Semantic cache ──────────────────────────────────────────────────
class SemanticCache:
    """Cache full answers keyed by question embedding. Handles paraphrasing."""

    def __init__(self, threshold=0.92, ttl_seconds=60):
        self.threshold  = threshold
        self.ttl        = ttl_seconds
        self.entries    = []
        self.hits       = 0
        self.misses     = 0

    def lookup(self, question):
        if not self.entries:
            self.misses += 1
            return None
        q_emb  = embedder.encode(question, convert_to_tensor=True, show_progress_bar=False)
        embs   = np.array([e['emb'] for e in self.entries])
        scores = util.cos_sim(q_emb,
                              embedder.encode([e['question'] for e in self.entries],
                                              convert_to_tensor=True, show_progress_bar=False)
                              )[0].cpu().numpy()
        best   = int(np.argmax(scores))
        if (scores[best] >= self.threshold
                and time.time() - self.entries[best]['t'] < self.ttl):
            self.hits += 1
            return self.entries[best]['answer']
        self.misses += 1
        return None

    def store(self, question, answer):
        emb = embedder.encode(question, show_progress_bar=False)
        self.entries.append({'emb': emb, 'question': question,
                             'answer': answer, 't': time.time()})

    def stats(self):
        return {'hits': self.hits, 'misses': self.misses, 'entries': len(self.entries)}


# ── Layer 3: Negative cache ────────────────────────────────────────────────────
class NegativeCache:
    """Cache error results to avoid hammering a broken endpoint."""

    def __init__(self, ttl_seconds=15):
        self.ttl    = ttl_seconds
        self._store = {}

    def is_negative(self, key):
        entry = self._store.get(key)
        if entry and time.time() - entry['t'] < self.ttl:
            return entry['error']
        return None

    def record(self, key, error_result):
        self._store[key] = {'error': error_result, 't': time.time()}


# ── Demo: all three layers ────────────────────────────────────────────────────
sem_cache = SemanticCache(threshold=0.88, ttl_seconds=60)
neg_cache = NegativeCache(ttl_seconds=15)

def cached_tool_call(tool_name, args, question=None):
    """Demonstrate all three cache layers in sequence."""
    # Layer 2: semantic cache on the question
    if question:
        cached = sem_cache.lookup(question)
        if cached:
            return f'[Semantic cache HIT] {cached}'

    # Layer 3: negative cache on the args
    neg_key = f'{tool_name}::{json.dumps(args, sort_keys=True)}'
    err = neg_cache.is_negative(neg_key)
    if err:
        return f'[Negative cache HIT] Skipping known-bad call: {err["error"]}'

    # Layer 1: tool-level result cache (inside the tool)
    result = TOOL_REGISTRY[tool_name](**args)
    source = result.pop('_source', 'live')

    if 'error' in result:
        neg_cache.record(neg_key, result)  # negative cache the error
        return f'[{source.upper()}] ERROR — cached as negative: {result}'

    answer = str(result)
    if question:
        sem_cache.store(question, answer)
    return f'[{source.upper()}] {result}'


print('=== Three-Layer Cache Demo ===\n')

q1  = "What's the weather in NYC?"
q1b = 'NYC weather right now?'
q1c = "How's it looking in New York?"
q_bad = "What's the weather in MadeUpCity?"

calls = [
    (q1,  'get_weather', {'city': 'New York'},     'First call — live'),
    (q1b, 'get_weather', {'city': 'New York'},     'Paraphrase — semantic cache should hit'),
    (q1c, 'get_weather', {'city': 'New York'},     'Another paraphrase — semantic cache'),
    (q_bad, 'get_weather', {'city': 'MadeUpCity'}, 'Bad city — will error'),
    (q_bad, 'get_weather', {'city': 'MadeUpCity'}, 'Repeat bad — negative cache should stop it'),
]

for question, tool_name, args, note in calls:
    result = cached_tool_call(tool_name, args, question=question)
    print(f'  {note}')
    print(f'  Q: {question}')
    print(f'  → {result[:120]}')
    print()

print('Semantic cache stats:', sem_cache.stats())


---
## 9. Streaming Snapshot Pattern — When Even APIs Are Too Slow

REST APIs are request-response. For truly real-time data (stock tickers, IoT sensors,
live scores), asking once per user query doesn't scale.

**The pattern:**
```
  Background worker ──► subscribes to stream ──► updates in-memory snapshot
                                                         │
  User query ─────────────────────────────────► reads snapshot  (~1ms)
```

The tool reads from the snapshot, not from the stream.  
Your snapshot might be 200ms old — invisible to the user, but you've avoided
a network call on the critical path.  
**This is how trading systems, sports-score apps, and live dashboards stay fast.**

Patterns you'll encounter in production:
- **WebSockets** — bidirectional, low-latency (chat, dashboards)
- **Server-Sent Events (SSE)** — one-way push (price tickers)
- **Kafka / Redis Streams** — high-volume event ingestion
- **Webhooks** — "tell me when something changes" (vs. polling)


In [ ]:
class StreamSnapshot:
    """
    In-memory snapshot store updated by a background thread.
    In production: use Redis with a pub/sub subscriber.

    The background worker runs continuously, simulating a WebSocket or
    Kafka consumer. The tool function reads the latest snapshot with a
    single dict lookup — no network call on the hot path.
    """

    def __init__(self):
        self._lock     = threading.Lock()
        self._data     = {}
        self._updates  = 0
        self._running  = False

    def update(self, key, value):
        with self._lock:
            self._data[key]  = {'value': value, 'updated_at': time.time()}
            self._updates   += 1

    def read(self, key):
        with self._lock:
            entry = self._data.get(key)
        if not entry:
            return None
        age_ms = (time.time() - entry['updated_at']) * 1000
        return {'value': entry['value'], 'age_ms': round(age_ms, 1)}

    def start_worker(self, symbols, update_interval=0.05):
        """Background thread that simulates receiving stream events."""
        self._running = True

        def worker():
            base_prices = {s: 100.0 + random.uniform(-20, 20) for s in symbols}
            while self._running:
                for sym in symbols:
                    delta = random.uniform(-0.5, 0.5)
                    base_prices[sym] = max(1.0, base_prices[sym] + delta)
                    self.update(sym, round(base_prices[sym], 4))
                time.sleep(update_interval)

        t = threading.Thread(target=worker, daemon=True)
        t.start()
        return t

    def stop(self):
        self._running = False


# Simulate a stock price stream
snapshot = StreamSnapshot()
SYMBOLS  = ['TECHCO', 'CLOUDX', 'AILAB']
snapshot.start_worker(SYMBOLS, update_interval=0.02)
time.sleep(0.1)  # let worker populate the snapshot

def get_live_price(symbol):
    """Tool function: reads from snapshot — no network call."""
    entry = snapshot.read(symbol.upper())
    if not entry:
        return {'error': 'symbol_not_found',
                'user_message': f'{symbol} is not tracked in the price stream.',
                'fallback_suggestion': f'Available symbols: {SYMBOLS}'}
    return {'symbol': symbol.upper(), 'price': entry['value'],
            'data_age_ms': entry['age_ms']}


print('=== Streaming Snapshot Demo ===\n')
print('Background worker is continuously updating prices...')
print()

for _ in range(4):
    t_start = time.perf_counter()
    prices  = {s: get_live_price(s) for s in SYMBOLS}
    t_end   = time.perf_counter()
    print(f'  Read latency: {(t_end - t_start)*1000:.2f}ms  '
          f'(no network call — single dict lookup)')
    for sym, p in prices.items():
        print(f'    {sym}: ${p["price"]}  (data age: {p["data_age_ms"]}ms)')
    time.sleep(0.1)
    print()

snapshot.stop()
print(f'Total snapshot updates during demo: {snapshot._updates}')
print('Snapshot stopped.')
print()
print('Key insight: read latency < 1ms regardless of how frequently the stream updates.')
print('Freshness (update interval) and query latency are completely decoupled.')


---
## 10. Evaluation — 10 Questions Across All Source Types


In [ ]:
EVAL_SET = [
    # Vector only
    {'q': "What's our cancellation policy?",
     'key': 'refund', 'needs': 'vector', 'label': 'Cancellation policy (doc)'},
    {'q': 'What are the SLA guarantees for enterprise tier?',
     'key': '99.9%', 'needs': 'vector', 'label': 'SLA enterprise (doc)'},
    # SQL only
    {'q': 'How many shipped orders do we have?',
     'key': str(len([1 for _ in range(1)])), 'needs': 'sql', 'label': 'Shipped order count (DB)'},
    # API only
    {'q': 'Where is my package with tracking number FX00000007?',
     'key': 'Memphis', 'needs': 'api', 'label': 'Package location (API)'},
    {'q': 'Is it safe to ship from Chicago today?',
     'key': 'NOT recommended', 'needs': 'api', 'label': 'Weather safety (API)'},
    {'q': "What's the current USD/EUR exchange rate?",
     'key': '0.91', 'needs': 'api', 'label': 'FX rate (API)'},
    {'q': 'Is the dashboard service up right now?',
     'key': 'degraded', 'needs': 'api', 'label': 'Service status (API)'},
    # Hybrid
    {'q': 'Why is order FX00000007 late, and what is our refund policy for delays?',
     'key': 'delayed', 'needs': 'hybrid', 'label': 'Late order + policy (hybrid)'},
    {'q': 'Is it safe to ship from Los Angeles, and what is our weather protocol?',
     'key': 'LA', 'needs': 'hybrid', 'label': 'LA weather + ops policy (hybrid)'},
    {'q': 'Is the checkout API up and what does our SLA say about downtime?',
     'key': 'operational', 'needs': 'hybrid', 'label': 'Service status + SLA (hybrid)'},
]


def score_hybrid(question, key):
    result = hybrid_rag(question, verbose=False)
    return 1.0 if key.lower() in result['answer'].lower() else 0.0


def score_vector_only(question, key):
    docs = vector_retrieve(question, top_k=2)
    ctx  = ' '.join(d['text'] for d, _ in docs)
    return 1.0 if key.lower() in ctx.lower() else 0.0


print(f'{"Label":<35} {"Needs":<9} {"Vector-only":<14} {"Hybrid"}')
print('-' * 72)
vec_scores, hyb_scores = [], []

for item in EVAL_SET:
    vs = score_vector_only(item['q'], item['key'])
    hs = score_hybrid(item['q'], item['key'])
    vec_scores.append(vs)
    hyb_scores.append(hs)
    v = 'YES' if vs else 'NO'
    h = 'YES' if hs else 'NO'
    print(f'{item["label"]:<35} {item["needs"]:<9} {v:<14} {h}')

print()
print(f'Vector-only accuracy : {np.mean(vec_scores):.0%} ({int(sum(vec_scores))}/{len(vec_scores)})')
print(f'Hybrid    accuracy   : {np.mean(hyb_scores):.0%} ({int(sum(hyb_scores))}/{len(hyb_scores)})')
print(f'Improvement          : +{(np.mean(hyb_scores)-np.mean(vec_scores)):.0%}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

labels = [e['label'][:28] for e in EVAL_SET]
x = np.arange(len(EVAL_SET))
w = 0.38

axes[0].bar(x - w/2, vec_scores, w, color='#E53935', alpha=0.85, label='Vector-only')
axes[0].bar(x + w/2, hyb_scores, w, color='#2E7D32', alpha=0.85, label='Hybrid (vector+SQL+API)')
axes[0].set_xticks(x)
axes[0].set_xticklabels(labels, rotation=35, ha='right', fontsize=8)
axes[0].set_ylim(-0.05, 1.3)
axes[0].set_ylabel('Key phrase found in answer (1=yes, 0=no)')
axes[0].set_title('Per-Question: Vector-Only vs. Hybrid RAG', fontweight='bold')
axes[0].legend(fontsize=9)

# Zone shading
needs = [e['needs'] for e in EVAL_SET]
boundaries = {'vector':('#E53935',0.04), 'sql':('#1565C0',0.04),
              'api':('#F57F17',0.04), 'hybrid':('#2E7D32',0.04)}
prev, prev_label = 0, needs[0]
for i, n in enumerate(needs + [None]):
    if n != prev_label or i == len(needs):
        color, alpha = boundaries[prev_label]
        axes[0].axvspan(prev - 0.5, i - 0.5, alpha=alpha, color=color)
        mid = (prev + i) / 2 - 0.5
        axes[0].text(mid, 1.22, prev_label, ha='center', fontsize=7.5,
                     color=color, fontweight='bold')
        prev, prev_label = i, n

# Right: overall
systems = ['Vector\nOnly', 'Hybrid\n(+SQL+API)']
avgs    = [np.mean(vec_scores), np.mean(hyb_scores)]
clrs    = ['#E53935', '#2E7D32']
bars    = axes[1].bar(systems, avgs, color=clrs, alpha=0.85, width=0.5)
axes[1].set_ylim(0, 1.2)
axes[1].set_ylabel('Average accuracy')
axes[1].set_title('Overall Accuracy', fontweight='bold')
for bar, val in zip(bars, avgs):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.03,
                 f'{val:.0%}', ha='center', fontsize=14, fontweight='bold')

show_plot()
print('Vector-only answers document questions but is blind to DB facts and live API data.')
print('Hybrid covers all three source types via routing.')


---
## 11. Claude API — Real Tool Calling

Claude's tool use API:
1. Pass `tools` (list of tool specs in Anthropic format) with the user message
2. Claude returns a `tool_use` content block when it wants to call a tool
3. Your code executes the tool and sends back a `tool_result` message
4. Claude either calls more tools or writes the final answer

```python
# The turn structure:
messages = [
    {'role': 'user',      'content': question},
    {'role': 'assistant', 'content': [{'type':'tool_use','id':..., 'name':..., 'input':...}]},
    {'role': 'user',      'content': [{'type':'tool_result','tool_use_id':..., 'content':...}]},
    # Claude then responds with final text
]
```

Install: `pip install anthropic`  
Set key: `export ANTHROPIC_API_KEY=sk-ant-...`

**Model split:** router = `claude-haiku-4-5-20251001` (fast, cheap).  
Tool selection + synthesis = `claude-sonnet-4-6` (better reasoning across tool results).


In [ ]:
ANTHROPIC_AVAILABLE = False
try:
    import anthropic
    ANTHROPIC_AVAILABLE = bool(os.environ.get('ANTHROPIC_API_KEY'))
except ImportError:
    pass

CLAUDE_TOOLS = [
    {
        'name': 'get_shipping_status',
        'description': (
            'Get the CURRENT, LIVE shipping status and location for an order by tracking ID. '
            'Use when the user asks where their package is RIGHT NOW, about delivery status, '
            'whether a shipment is delayed, or its ETA. '
            'Do NOT use for historical order data or refund policies.'
        ),
        'input_schema': {
            'type': 'object',
            'properties': {
                'tracking_id': {'type': 'string',
                                'description': 'Carrier tracking number (e.g. FX00000007)'}
            },
            'required': ['tracking_id']
        }
    },
    {
        'name': 'get_weather',
        'description': (
            'Get current weather conditions for a city. Use when the user asks about '
            'today\'s weather, whether conditions are safe for shipping, or weather '
            'impact on warehouse operations. Do NOT use for historical weather.'
        ),
        'input_schema': {
            'type': 'object',
            'properties': {
                'city': {'type': 'string', 'description': 'City name (e.g. Chicago)'}
            },
            'required': ['city']
        }
    },
    {
        'name': 'get_exchange_rate',
        'description': (
            'Get the CURRENT exchange rate between two currencies. '
            'Use when the user needs to convert amounts or asks for today\'s rate. '
            'Supported pairs: USD/EUR, USD/GBP, USD/JPY.'
        ),
        'input_schema': {
            'type': 'object',
            'properties': {
                'pair': {'type': 'string', 'description': 'Currency pair as FROM/TO, e.g. USD/EUR'}
            },
            'required': ['pair']
        }
    },
    {
        'name': 'get_service_status',
        'description': (
            'Check if a service is currently UP or DOWN. '
            'Use when the user asks if a service is working, reports an outage, or asks '
            'about system health. Services: "api", "dashboard", "checkout". '
            'Do NOT use for planned maintenance windows.'
        ),
        'input_schema': {
            'type': 'object',
            'properties': {
                'service': {'type': 'string',
                            'description': 'Service name: api, dashboard, or checkout'}
            },
            'required': ['service']
        }
    },
]

SYSTEM_PROMPT = """\
You are a helpful customer support assistant with access to live tools.
When you call a tool and it returns an error field, tell the user clearly
and do NOT fabricate a value. Never invent shipping locations, prices, or
service statuses. If you cannot answer from tool results, say so.
"""

if ANTHROPIC_AVAILABLE:
    client = anthropic.Anthropic()

    def claude_tool_loop(question, max_turns=5):
        """Multi-turn ReAct loop using Claude's native tool use."""
        messages = [{'role': 'user', 'content': question}]

        for turn in range(max_turns):
            resp = client.messages.create(
                model='claude-sonnet-4-6',
                max_tokens=1024,
                system=SYSTEM_PROMPT,
                tools=CLAUDE_TOOLS,
                messages=messages,
            )

            # Append Claude's response to the conversation
            messages.append({'role': 'assistant', 'content': resp.content})

            if resp.stop_reason == 'end_turn':
                # Extract text from the final response
                for block in resp.content:
                    if hasattr(block, 'text'):
                        return block.text
                return str(resp.content)

            if resp.stop_reason == 'tool_use':
                tool_results = []
                for block in resp.content:
                    if block.type == 'tool_use':
                        tool_name = block.name
                        args      = block.input
                        print(f'  [Claude calls] {tool_name}({args})')

                        if tool_name in TOOL_REGISTRY:
                            result = TOOL_REGISTRY[tool_name](**args)
                            result.pop('_source', None)
                        else:
                            result = {'error': 'unknown_tool',
                                      'user_message': f'{tool_name} is not available.'}

                        print(f'  [Tool result]  {result}')
                        tool_results.append({
                            'type': 'tool_result',
                            'tool_use_id': block.id,
                            'content': json.dumps(result),
                        })

                messages.append({'role': 'user', 'content': tool_results})

        return 'Max turns reached without a final answer.'


    claude_qs = [
        'Where is my package FX00000007?',
        'Is it safe to ship from Chicago today?',
        'Is the dashboard service currently up?',
    ]
    print('=== Claude Tool Calling ===\n')
    for q in claude_qs:
        print(f'Q: {q}')
        answer = claude_tool_loop(q)
        print(f'A: {answer}')
        print()

else:
    print('anthropic not installed or ANTHROPIC_API_KEY not set.')
    print()
    print('To enable:')
    print('  pip install anthropic')
    print('  export ANTHROPIC_API_KEY=sk-ant-...')
    print()
    print('Key points about Claude tool calling:')
    print('  1. Pass CLAUDE_TOOLS list in the client.messages.create() call')
    print('  2. Check resp.stop_reason == "tool_use" to detect a tool call')
    print('  3. Execute the tool, append tool_result to messages, call Claude again')
    print('  4. Loop until stop_reason == "end_turn"')
    print()
    print('Model recommendations:')
    print('  Routing only      : claude-haiku-4-5-20251001  (~10x cheaper, <100ms)')
    print('  Tool use + answer : claude-sonnet-4-6          (better at multi-tool reasoning)')
    print()
    print('Critical rule: include in SYSTEM_PROMPT:')
    print('  "If a tool returns an error, tell the user. Do NOT fabricate a value."')
    print('  Without this, Claude may invent shipping statuses when the API fails.')


---
## Key Takeaways

1. **There are three kinds of knowledge, not two.** Vector search gives you what was true
   when you indexed. SQL gives you what's true in your DB now. APIs give you what's true
   in the world right now. They are not interchangeable.

2. **The router must recognise temporal urgency.** Questions about *right now* — current
   weather, live tracking, today's rate — are the API lane. The signal is the word "now"
   (often implicit: "where is my package", not "where was my package").

3. **Tool descriptions are the model's decision tree.** The LLM picks tools based on
   descriptions alone. Vague descriptions = wrong tools called. Write them like product
   copy: say *when to use* and optionally *when NOT to use*.

4. **A production tool wrapper has five layers.** Validation → cache → retry → response
   shaping → error formatting. Don't return raw API JSON. Don't return
   `ECONNREFUSED` errors. Shape responses and make errors actionable for the LLM.

5. **Never let an API error become an LLM hallucination.** If the tool fails, it must
   return `{"error": "..."}` and your system prompt must say: *"If a tool returned an
   error, say so. Do not fabricate a value."* This is the single most important rule
   of API-RAG.

6. **Implement all three cache layers.** Result cache (per-args TTL), semantic cache
   (per-question embedding similarity), negative cache (don't retry known-bad inputs).
   A 60-second semantic cache typically cuts API costs by 90%+ with no user-visible
   degradation.

7. **For truly real-time data, decouple freshness from query latency.** Use the streaming
   snapshot pattern: a background worker keeps the snapshot fresh; your tool reads
   from it in < 1ms. This is how trading systems stay fast without hammering APIs.

8. **API-RAG is your most fragile pipeline branch.** Every tool call is a dependency
   on someone else's uptime. When in doubt, sync nightly into Postgres and use SQL.
   Add API tools only when the freshness benefit clearly justifies the operational cost.

---

*Up next — Lesson 9.4: My documents have tables, charts, and diagrams.*
*How do I RAG over content that isn't just text?*
